# Hands-On Lab P2W1D5 — Reliability Layer: Before / After

**Module 21 · Function Calling & Validation**  
**Durasi:** 120 menit · **Level:** beginner–intermediate · **Bahasa:** Indonesia

Anda melanjutkan asisten kedai dari Module 20. Pertanyaan kebijakan dijawab dari FAQ lokal; status pesanan dibaca melalui tool LangChain. Jalankan versi **before** yang sengaja sederhana pada data sintetis, amati kegagalannya, lalu jalankan versi **after** pada input dan gangguan yang sama.

> **Security Note:** Versi before sengaja memperlihatkan akses yang keliru hanya pada pesanan sintetis. Jangan pakai pola itu dengan data pelanggan sungguhan.

## Tujuan, hasil kerja, dan waktu

Setelah lab, Anda dapat (1) membedakan permintaan tool oleh model dari eksekusi oleh aplikasi; (2) menunjukkan bagaimana argumen dan ownership diperiksa sebelum handler; (3) membatasi retry hanya untuk error sementara; dan (4) memberi fallback jujur setelah kegagalan berulang.

**Artefak:** notebook terisi, tabel before/after, hasil dua pertanyaan live, satu modifikasi mandiri, dan refleksi singkat.

| Kegiatan | Menit |
|---|---:|
| Setup dan FAQ | 15 |
| Before: amati kegagalan | 20 |
| After: kontrak dan ownership | 30 |
| Retry dan fallback | 20 |
| Integrasi LangChain live | 20 |
| Modifikasi mandiri dan review | 15 |
| **Total** | **120** |

**Prasyarat:** Python 3.11+, dasar Python, dan pengantar LangChain Module 20. OpenAI API key serta internet diperlukan untuk dua demo live. Pemakaian API dapat dikenai biaya.

## Skenario dan peta alur

Pelanggan menanyakan kebijakan penggantian atau status pesanan. FAQ berisi kebijakan umum. Status hanya boleh datang dari backend order dan hanya boleh dilihat pemiliknya.

```text
Pertanyaan → FAQ retrieval → model LangChain
                              ├─ jawaban kebijakan dari FAQ
                              └─ tool call + call ID → aplikasi
                                                    ├─ BEFORE: langsung panggil service
                                                    └─ AFTER: allowlist → schema → owner
                                                               → retry terbatas → ToolMessage
```

**Prediksi sebelum menjalankan:** jika model mengirim ID pesanan milik orang lain, komponen mana yang seharusnya menghentikan akses? Model, schema, atau aplikasi?

## 1. Setup dan FAQ lokal

Instal dependensi dan pilih kernel sesuai `README.md`. Jalankan cell berurutan. Key baru diminta pada bagian live; perbandingan sintetis awal dapat dijalankan tanpa API key.

In [1]:
import getpass
import json
import os
import re
from datetime import datetime, timezone
from typing import Any, Literal

from IPython.display import Markdown, display
from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

print("Impor berhasil. Python:", os.sys.version.split()[0])

Impor berhasil. Python: 3.10.20


In [2]:
FAQ_DOCUMENTS = [
    Document(
        page_content=(
            "Jika minuman dibuat dengan keliru, pelanggan dapat meminta penggantian "
            "dalam 10 menit setelah pesanan diterima. Tunjukkan struk pesanan."
        ),
        metadata={"source": "kebijakan-penggantian"},
    ),
    Document(
        page_content="Kedai buka setiap hari pukul 08.00 sampai 20.00 waktu setempat.",
        metadata={"source": "jam-operasional"},
    ),
    Document(
        page_content=(
            "Pesanan yang belum mulai dibuat dapat diperbarui dengan menghubungi kasir. "
            "Status pesanan tertentu harus diperiksa melalui nomor pesanan."
        ),
        metadata={"source": "perubahan-pesanan"},
    ),
]

STOP_WORDS = {
    "apa", "berapa", "dan", "di", "dengan", "dapat", "untuk", "yang",
    "saya", "tolong", "bagaimana", "kah", "ini", "itu", "ada",
}

def retrieve_faq(question: str, limit: int = 2) -> list[Document]:
    """Return the FAQ documents with the most overlapping meaningful words."""
    terms = {
        word
        for word in re.findall(r"[a-z0-9]+", question.casefold())
        if word not in STOP_WORDS
    }
    ranked = []
    for position, document in enumerate(FAQ_DOCUMENTS):
        document_terms = set(
            re.findall(r"[a-z0-9]+", document.page_content.casefold())
        )
        score = len(terms & document_terms)
        if score:
            ranked.append((-score, position, document))
    ranked.sort(key=lambda item: (item[0], item[1]))
    return [item[2] for item in ranked[:limit]]

def format_documents(documents: list[Document]) -> str:
    return "\n\n".join(document.page_content for document in documents)

faq_context_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        documents=RunnableLambda(retrieve_faq),
    )
    | RunnablePassthrough.assign(
        context=lambda values: format_documents(values["documents"])
    )
)

print("FAQ siap:", len(FAQ_DOCUMENTS), "dokumen")

FAQ siap: 3 dokumen


In [6]:
faq_preview = faq_context_chain.invoke(
    "Berapa lama minuman yang salah dibuat dapat diganti dan apa syaratnya?"
)
print("Pertanyaan:", faq_preview["question"])
for document in faq_preview["documents"]:
    print("-", document.metadata["source"], "→", document.page_content)
print("\nKonteks yang dikirim ke model:\n", faq_preview["context"])

Pertanyaan: Berapa lama minuman yang salah dibuat dapat diganti dan apa syaratnya?
- kebijakan-penggantian → Jika minuman dibuat dengan keliru, pelanggan dapat meminta penggantian dalam 10 menit setelah pesanan diterima. Tunjukkan struk pesanan.
- perubahan-pesanan → Pesanan yang belum mulai dibuat dapat diperbarui dengan menghubungi kasir. Status pesanan tertentu harus diperiksa melalui nomor pesanan.

Konteks yang dikirim ke model:
 Jika minuman dibuat dengan keliru, pelanggan dapat meminta penggantian dalam 10 menit setelah pesanan diterima. Tunjukkan struk pesanan.

Pesanan yang belum mulai dibuat dapat diperbarui dengan menghubungi kasir. Status pesanan tertentu harus diperiksa melalui nomor pesanan.


> **Checkpoint 1 — FAQ:** dokumen `kebijakan-penggantian` terambil dan menyebut batas **10 menit** serta **struk**. Jika kosong, cek kata pada pertanyaan dan isi FAQ. Retriever ini hanya mencocokkan kata; hasilnya bukan bukti bahwa jawaban model selalu benar.

## 2. BEFORE — tool call langsung ke service

Service berikut berisi tiga order sintetis dan mode kegagalan yang dapat diatur. Identitas sesi `customer-001` berasal dari aplikasi. Fungsi `naive_dispatch` mewakili pola awal yang langsung memakai argumen model: tidak ada kontrak ketat, pemeriksaan pemilik, atau retry.

> **Common Mistake:** Schema tool yang ditunjukkan ke model tidak menggantikan pemeriksaan di aplikasi. Permintaan tool tetap input tidak tepercaya.

In [7]:
AUTHENTICATED_SESSION = {"user_id": "customer-001"}
ORDERS = {
    "ORD-100101": {"owner_id": "customer-001", "status": "processing", "last_updated": datetime(2026, 10, 1, 8, 30, tzinfo=timezone.utc)},
    "ORD-100102": {"owner_id": "customer-001", "status": "delivered", "last_updated": datetime(2026, 9, 30, 12, 15, tzinfo=timezone.utc)},
    "ORD-200202": {"owner_id": "customer-002", "status": "shipped", "last_updated": datetime(2026, 10, 1, 7, 0, tzinfo=timezone.utc)},
}
ORDER_OWNER_INDEX = {order_id: row["owner_id"] for order_id, row in ORDERS.items()}

class TransientOrderServiceError(Exception):
    '''Gangguan sementara yang boleh diulang.'''

class PermanentOrderServiceError(Exception):
    '''Kegagalan permanen yang tidak boleh diulang.'''

class DemoOrderService:
    '''Backend lokal read-only; mode gagal membuat latihan berulang secara konsisten.'''

    def __init__(self, records: dict[str, dict[str, Any]]) -> None:
        self.records = records
        self.calls = 0
        self.reset()

    def reset(self, *, failures: int = 0, permanent: bool = False, bad_result: dict[str, Any] | None = None) -> None:
        self.failures_remaining = failures
        self.permanent_failure = permanent
        self.bad_result = bad_result

    def get_status(self, order_id: str) -> dict[str, Any]:
        self.calls += 1
        if self.permanent_failure:
            raise PermanentOrderServiceError("Simulasi error permanen")
        if self.failures_remaining > 0:
            self.failures_remaining -= 1
            raise TransientOrderServiceError("Simulasi gangguan sementara")
        if self.bad_result is not None:
            return self.bad_result
        row = self.records.get(order_id)
        if row is None:
            raise PermanentOrderServiceError("Order tidak ada")
        return {"order_id": order_id, "status": row["status"], "last_updated": row["last_updated"]}

order_service = DemoOrderService(ORDERS)

def naive_dispatch(tool_name: str, raw_arguments: object) -> dict[str, Any]:
    '''Contoh sengaja lemah: argumen model diteruskan tanpa kontrol akses.'''
    if tool_name != "get_order_status":
        raise KeyError(tool_name)
    arguments = json.loads(raw_arguments) if isinstance(raw_arguments, str) else raw_arguments
    return order_service.get_status(arguments["order_id"])

print("Sesi:", AUTHENTICATED_SESSION["user_id"], "| Order sintetis:", len(ORDERS))

Sesi: customer-001 | Order sintetis: 3


### Amati masalah sebelum memperbaiki

Jalankan empat input di bawah. Catat kapan handler dipanggil, apakah error mentah muncul, dan apakah order milik `customer-002` bocor. Semua pengujian memakai data sintetis.

In [8]:
BEFORE_INPUTS = [
    ("JSON rusak", "get_order_status", "{"),
    ("field tambahan", "get_order_status", {"order_id": "ORD-100101", "user_id": "customer-001"}),
    ("ID tidak valid", "get_order_status", {"order_id": "100101"}),
    ("order pengguna lain", "get_order_status", {"order_id": "ORD-200202"}),
]

def observe_before(tool_name: str, arguments: object, *, failures: int = 0, permanent: bool = False) -> dict[str, Any]:
    order_service.reset(failures=failures, permanent=permanent)
    calls_before = order_service.calls
    try:
        result = naive_dispatch(tool_name, arguments)
        outcome = f"status={result['status']}"
    except (json.JSONDecodeError, KeyError, TypeError, TransientOrderServiceError, PermanentOrderServiceError) as error:
        outcome = type(error).__name__
    return {"outcome": outcome, "calls": order_service.calls - calls_before}

before_rows = []
for label, name, arguments in BEFORE_INPUTS:
    observed = observe_before(name, arguments)
    before_rows.append((label, observed))
    print(f"{label:20} → {observed['outcome']:31} | handler={observed['calls']}")

JSON rusak           → JSONDecodeError                 | handler=0
field tambahan       → status=processing               | handler=1
ID tidak valid       → PermanentOrderServiceError      | handler=1
order pengguna lain  → status=shipped                  | handler=1


> **Checkpoint 2 — Before:** field tambahan diterima dan handler berjalan; ID salah tetap mencapai handler; order pengguna lain menghasilkan `status=shipped`. JSON rusak gagal sebelum handler, tetapi tanpa pesan aman. Jelaskan mengapa setiap hasil ini tidak cukup untuk aplikasi pelanggan nyata.

## 3. AFTER — validasi sebelum handler

Empat keputusan kini dilakukan oleh aplikasi: nama tool harus diizinkan, argumen harus sesuai kontrak Pydantic v2, order harus ada dan dimiliki identitas sesi, lalu service dipanggil. `user_id` tidak boleh datang dari model. ID tidak dikenal dan ID milik orang lain memakai pesan akses yang sama.

In [9]:
class OrderStatusArgs(BaseModel):
    '''Argumen yang boleh diminta model.'''

    model_config = ConfigDict(extra="forbid", strict=True)
    order_id: str = Field(pattern=r"^ORD-[0-9]{6}$", description="Contoh: ORD-100101")

class SafeToolError(Exception):
    '''Error aman untuk pelanggan; rincian backend tidak dikirim ke model.'''

    def __init__(self, public_message: str, outcome: str, attempts: int = 0) -> None:
        super().__init__(public_message)
        self.public_message = public_message
        self.outcome = outcome
        self.attempts = attempts

FALLBACK_MESSAGE = "Status pesanan belum bisa diperiksa saat ini. Silakan coba lagi beberapa saat lagi."
MAX_ATTEMPTS = 3
TOOL_HANDLERS = {"get_order_status": order_service.get_status}

def parse_order_args(raw_arguments: object) -> OrderStatusArgs:
    if isinstance(raw_arguments, str):
        try:
            raw_arguments = json.loads(raw_arguments)
        except json.JSONDecodeError as error:
            raise SafeToolError("Argumen tool tidak valid.", "invalid_arguments") from error
    if not isinstance(raw_arguments, dict):
        raise SafeToolError("Argumen tool tidak valid.", "invalid_arguments")
    try:
        return OrderStatusArgs.model_validate(raw_arguments)
    except ValidationError as error:
        raise SafeToolError("Argumen tool tidak sesuai kontrak.", "invalid_arguments") from error

def ensure_order_owner(order_id: str, user_id: str) -> None:
    if ORDER_OWNER_INDEX.get(order_id) != user_id:
        raise SafeToolError("Pesanan tidak ditemukan atau tidak dapat diakses.", "access_denied")

def call_with_retry(handler, order_id: str) -> tuple[dict[str, Any], int]:
    '''Retry hanya gangguan sementara; tanpa jeda agar simulasi cepat.'''
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return handler(order_id), attempt
        except TransientOrderServiceError as error:
            if attempt == MAX_ATTEMPTS:
                raise SafeToolError(FALLBACK_MESSAGE, "retry_exhausted", attempt) from error
    raise RuntimeError("Jalur retry tidak terjangkau")

def dispatch_tool_call(tool_name: str, raw_arguments: object, *, user_id: str) -> tuple[dict[str, Any], int]:
    '''Jalankan allowlist → schema → owner → handler dengan retry terbatas.'''
    handler = TOOL_HANDLERS.get(tool_name)
    if handler is None:
        raise SafeToolError("Tool tidak tersedia.", "unknown_tool")
    args = parse_order_args(raw_arguments)
    ensure_order_owner(args.order_id, user_id)
    try:
        return call_with_retry(handler, args.order_id)
    except PermanentOrderServiceError as error:
        raise SafeToolError("Status pesanan tidak dapat diperiksa.", "permanent_failure", 1) from error

print("Allowlist:", list(TOOL_HANDLERS), "| maksimum percobaan:", MAX_ATTEMPTS)

Allowlist: ['get_order_status'] | maksimum percobaan: 3


### Bandingkan input yang sama

Cell berikut menjalankan empat input before lagi melalui dispatcher after. Kolom `handler` menunjukkan apakah validasi terjadi sebelum service. Lalu uji nama tool yang tidak diizinkan.

In [10]:
def observe_after(tool_name: str, arguments: object, *, failures: int = 0, permanent: bool = False) -> dict[str, Any]:
    order_service.reset(failures=failures, permanent=permanent)
    calls_before = order_service.calls
    try:
        result, attempts = dispatch_tool_call(tool_name, arguments, user_id=AUTHENTICATED_SESSION["user_id"])
        return {"outcome": f"status={result['status']}", "calls": order_service.calls - calls_before, "attempts": attempts}
    except SafeToolError as error:
        return {"outcome": error.outcome, "calls": order_service.calls - calls_before, "attempts": error.attempts, "message": error.public_message}

comparison_rows = []
for (label, before), (_, name, arguments) in zip(before_rows, BEFORE_INPUTS):
    after = observe_after(name, arguments)
    comparison_rows.append((label, before, after))

table = ["| Kasus | Before | Handler | After | Handler |", "|---|---|---:|---|---:|"]
for label, before, after in comparison_rows:
    table.append(f"| {label} | {before['outcome']} | {before['calls']} | {after['outcome']} | {after['calls']} |")
display(Markdown("\n".join(table)))

assert all(after["calls"] == 0 for _, _, after in comparison_rows)
assert comparison_rows[3][1]["outcome"] == "status=shipped"
print("Tool tidak dikenal:", observe_after("delete_order", {"order_id": "ORD-100101"}))

| Kasus | Before | Handler | After | Handler |
|---|---|---:|---|---:|
| JSON rusak | JSONDecodeError | 0 | invalid_arguments | 0 |
| field tambahan | status=processing | 1 | invalid_arguments | 0 |
| ID tidak valid | PermanentOrderServiceError | 1 | invalid_arguments | 0 |
| order pengguna lain | status=shipped | 1 | access_denied | 0 |

Tool tidak dikenal: {'outcome': 'unknown_tool', 'calls': 0, 'attempts': 0, 'message': 'Tool tidak tersedia.'}


> **Checkpoint 3 — After:** keempat input ditolak tanpa memanggil handler. `delete_order` juga ditolak. **Bandingkan:** schema menolak bentuk argumen, ownership menolak akses meski ID valid, dan allowlist menolak nama tool. Semua pemeriksaan terjadi sebelum service.

## 4. BEFORE / AFTER — error sementara, permanen, dan fallback

Service disetel ulang sebelum setiap sisi dibandingkan. Before berhenti pada error pertama. After mencoba ulang `TransientOrderServiceError` sampai tiga kali. Error permanen tidak diulang. Jika tiga percobaan tetap gagal, aplikasi mengembalikan pesan fallback tanpa menebak status. Simulasi tidak memakai jeda agar output stabil; produksi memerlukan timeout dan backoff.

In [11]:
ORDER_REQUEST = {"order_id": "ORD-100101"}
failure_cases = [
    ("2 error sementara", {"failures": 2}),
    ("error permanen", {"permanent": True}),
    ("gangguan berulang", {"failures": 99}),
]
failure_rows = []
for label, settings in failure_cases:
    before = observe_before("get_order_status", ORDER_REQUEST, **settings)
    after = observe_after("get_order_status", ORDER_REQUEST, **settings)
    failure_rows.append((label, before, after))

table = ["| Gangguan | Before | Panggilan | After | Panggilan |", "|---|---|---:|---|---:|"]
for label, before, after in failure_rows:
    table.append(f"| {label} | {before['outcome']} | {before['calls']} | {after['outcome']} | {after['calls']} |")
display(Markdown("\n".join(table)))

assert failure_rows[0][2]["outcome"] == "status=processing" and failure_rows[0][2]["calls"] == 3
assert failure_rows[1][2]["outcome"] == "permanent_failure" and failure_rows[1][2]["calls"] == 1
assert failure_rows[2][2]["outcome"] == "retry_exhausted" and failure_rows[2][2]["calls"] == 3
print("Fallback:", failure_rows[2][2]["message"])
order_service.reset()

| Gangguan | Before | Panggilan | After | Panggilan |
|---|---|---:|---|---:|
| 2 error sementara | TransientOrderServiceError | 1 | status=processing | 3 |
| error permanen | PermanentOrderServiceError | 1 | permanent_failure | 1 |
| gangguan berulang | TransientOrderServiceError | 1 | retry_exhausted | 3 |

Fallback: Status pesanan belum bisa diperiksa saat ini. Silakan coba lagi beberapa saat lagi.


> **Checkpoint 4 — Retry:** dua gangguan sementara pulih pada percobaan ke-3; error permanen hanya satu pemanggilan; gangguan berulang berhenti pada tiga pemanggilan dan menampilkan fallback. Validasi dan akses ditolak sebelum handler, jadi tidak masuk jalur retry.

**Diskusi singkat:** mengapa mengulang error authorization tiga kali tidak membantu? Apa yang berubah jika handler melakukan aksi tulis dan request dikirim ulang?

## 5. Hubungkan ke LangChain dan model live

Bagian ini memakai model live. Key dibaca dari environment atau diminta dengan input tersembunyi. Tool LangChain menyediakan schema kepada model, tetapi aplikasi tetap memanggil dispatcher sendiri agar `tool_call_id`, validasi, dan hasil dapat diperiksa. Untuk pertanyaan kebijakan, aplikasi hanya memberi FAQ. Untuk status pesanan, aplikasi mewajibkan satu tool call; jawaban akhir memakai hasil tool.

In [18]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (input tersembunyi): ")

MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-6-luna")
model = ChatOpenAI(model=MODEL_NAME, timeout=20)
print("Model siap:", MODEL_NAME, "| key tidak ditampilkan atau disimpan")

Model siap: gpt-6-luna | key tidak ditampilkan atau disimpan


In [19]:
@tool("get_order_status", args_schema=OrderStatusArgs)
def get_order_status(order_id: str) -> dict[str, Any]:
    '''Baca status satu pesanan milik pelanggan yang sedang login.

    Args:
        order_id: Nomor pesanan, contoh ORD-100101.
    '''
    result, _attempts = dispatch_tool_call(
        "get_order_status", {"order_id": order_id}, user_id=AUTHENTICATED_SESSION["user_id"]
    )
    return result

model_requiring_tool = model.bind_tools([get_order_status], tool_choice="required", parallel_tool_calls=False)
SYSTEM_PROMPT = (
    "Kamu asisten kedai. Jawab dalam Bahasa Indonesia. "
    "Untuk kebijakan, gunakan hanya FAQ yang diberikan. "
    "Jangan menebak status pesanan; gunakan hasil get_order_status. "
    "Jika tool gagal, katakan status belum dapat diperiksa."
)

def question_messages(question: str) -> tuple[list, list[Document]]:
    retrieval = faq_context_chain.invoke(question)
    context = retrieval["context"] or "Tidak ada FAQ yang cocok."
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Pertanyaan pelanggan: {question}\n\nKonteks FAQ:\n{context}"),
    ]
    return messages, retrieval["documents"]

print("Schema tool terikat:", get_order_status.name)

Schema tool terikat: get_order_status


### Pertanyaan kebijakan: FAQ tanpa order tool

Jalankan pertanyaan berikut. Aplikasi tidak memberikan tool order untuk jalur kebijakan. Inspeksi dokumen yang diambil dan cek apakah jawaban menyebut **10 menit** serta **struk**. Kata-kata jawaban model dapat bervariasi.

In [20]:
policy_messages, policy_documents = question_messages(
    "Berapa lama minuman yang salah dibuat dapat diganti dan apa syaratnya?"
)
policy_response = model.invoke(policy_messages)
print("FAQ sources:", [document.metadata["source"] for document in policy_documents])
print("Tool calls:", len(policy_response.tool_calls))
print("Jawaban:", policy_response.content)
assert len(policy_response.tool_calls) == 0

FAQ sources: ['kebijakan-penggantian', 'perubahan-pesanan']
Tool calls: 0
Jawaban: Minuman yang salah dibuat dapat diganti jika Anda memintanya dalam 10 menit setelah pesanan diterima. Mohon tunjukkan struk pesanan.


> **Checkpoint 5 — FAQ:** `kebijakan-penggantian` terambil, tool call = 0, dan jawaban harus dapat ditelusuri ke teks FAQ. Jika model menambah klaim lain, tandai klaim itu sebagai tidak didukung.

### Pertanyaan order: inspeksi request, dispatcher, dan ToolMessage

Cell ini meminta model membuat **satu** tool call untuk order sintetis. Aplikasi memeriksa nama dan argumen, menjalankan service, lalu membuat `ToolMessage` dengan ID yang sama. Jika tool gagal, aplikasi langsung menampilkan fallback; model tidak diminta mengarang status.

In [23]:
order_service.reset()
messages, order_documents = question_messages("Tolong cek status pesanan ORD-200202.")
request = model_requiring_tool.invoke(messages)
print("Tool request:", request.tool_calls)

if len(request.tool_calls) != 1:
    print("Jawaban aman:", FALLBACK_MESSAGE)
else:
    call = request.tool_calls[0]
    try:
        result, attempts = dispatch_tool_call(
            call["name"], call["args"], user_id=AUTHENTICATED_SESSION["user_id"]
        )
        tool_message = ToolMessage(
            content=json.dumps({"ok": True, "result": result}, default=str, ensure_ascii=False),
            tool_call_id=call["id"],
        )
        print("Status backend:", result["status"], "| percobaan:", attempts)
        print("Request ID:", call["id"], "| Result ID:", tool_message.tool_call_id)
        assert tool_message.tool_call_id == call["id"]
        final_response = model.invoke(messages + [request, tool_message])
        print("Jawaban:", final_response.content)
    except SafeToolError as error:
        tool_message = ToolMessage(
            content=json.dumps({"ok": False, "error": error.public_message}, ensure_ascii=False),
            tool_call_id=call["id"],
        )
        print("Request ID:", call["id"], "| Result ID:", tool_message.tool_call_id)
        print("Jawaban aman:", error.public_message)

Tool request: [{'name': 'get_order_status', 'args': {'order_id': 'ORD-200202'}, 'id': 'call_GPplHigMYx5vQGz1VdUhN6Uz', 'type': 'tool_call'}]
Request ID: call_GPplHigMYx5vQGz1VdUhN6Uz | Result ID: call_GPplHigMYx5vQGz1VdUhN6Uz
Jawaban aman: Pesanan tidak ditemukan atau tidak dapat diakses.


> **Checkpoint 6 — Function calling:** permintaan bernama `get_order_status`, `ToolMessage.tool_call_id` sama dengan ID permintaan, dan status sintetis `processing` berasal dari backend. Jika model mengirim argumen berbeda, dispatcher tetap menjaga keamanan; lihat bagian troubleshooting.

## 6. Tantangan mandiri dan opsi lanjut

**Wajib (10 menit):** Tambahkan order sintetis `ORD-100103` untuk `customer-001` dengan status `pending`. Jalankan `observe_after` untuk sesi `customer-001` dan `customer-002`. Tulis dua hasil, jumlah pemanggilan handler, dan alasan mengapa `user_id` tidak diambil dari argumen model. Gunakan `ORDER_OWNER_INDEX` yang konsisten dengan data baru. Jangan tampilkan solusi lengkap sebelum mencoba.

**Opsional — validasi hasil tool:** Coba `order_service.reset(bad_result={"order_id": "ORD-100101", "status": "teleported", "last_updated": datetime.now(timezone.utc)})`. Saat ini dispatcher inti meneruskan hasil mentah. Buat `OrderStatusResult` dengan Pydantic v2 dan status yang diizinkan (`pending`, `processing`, `shipped`, `delivered`, `cancelled`); validasi hasil **sebelum** membuat `ToolMessage`, lalu ubah error menjadi pesan aman. Pastikan mode service di-reset sesudah eksperimen. Ini adalah langkah tambahan, bukan syarat lulus.

> **Production Note:** Latihan ini memakai service in-memory, identitas sesi sintetis, retry langsung tanpa backoff, dan FAQ berbasis kata. Sistem produksi membutuhkan autentikasi nyata, otorisasi dekat sumber data, timeout dan backoff, pemantauan, serta evaluasi jawaban RAG. Tool yang mengubah data memerlukan desain tambahan yang berbeda dari tool baca ini.

## Troubleshooting, pengumpulan, dan rubric

| Gejala | Penyebab umum | Langkah |
|---|---|---|
| `ModuleNotFoundError` | Kernel berbeda dari environment instalasi | Pilih kernel `.venv`, lalu jalankan ulang impor |
| API key error | Key kosong atau akses model tidak tersedia | Isi key melalui prompt tersembunyi; periksa `OPENAI_MODEL` |
| Model tidak memberi satu tool call | Model atau endpoint tidak mendukung tool choice | Pakai model default dan lihat `request.tool_calls`; jangan mengarang status |
| Order baru ditolak | `ORDERS` dan `ORDER_OWNER_INDEX` tidak selaras | Perbarui keduanya, lalu jalankan ulang cell definisi |
| Retry tidak terlihat | `failures` belum disetel atau cell dijalankan tidak berurutan | Jalankan ulang tabel gangguan dari awal |

**Kumpulkan:** notebook tanpa API key atau output rahasia, tabel before/after, catatan hasil FAQ dan order live, hasil tantangan mandiri, serta jawaban singkat: (1) guard mana yang bekerja sebelum handler? (2) error mana yang diulang? (3) mengapa fallback tidak menyebut status?

| Rubric | Bobot | Bukti |
|---|---:|---|
| Perbandingan before/after | 25 | Tabel dan penjelasan perubahan hasil |
| Validasi, allowlist, ownership | 30 | Input buruk dan lintas pengguna berhenti sebelum handler |
| Retry dan fallback | 25 | Tiga skenario gangguan sesuai batas percobaan |
| LangChain dan FAQ | 15 | FAQ tanpa tool; order dengan ToolMessage ber-ID sama |
| Refleksi | 5 | Alasan keamanan dan batas contoh jelas |

**Panduan instruktur:** buka dengan prediksi kasus order pengguna lain, jalankan before tanpa menyebut hasilnya, lalu minta peserta menyebut guard yang hilang. Saat review, tekankan bahwa schema tidak membuktikan ownership dan retry tidak memperbaiki error permanen. Jika API live bermasalah, gunakan tabel sintetis sebagai bukti inti dan demonstrasikan bentuk `tool_calls`/`ToolMessage` dari kode tanpa mengarang output live.